# bmkb-observability — Drive & Observe: Agentic RAG on a fully-managed Bedrock KB

This notebook is the **post-deploy driver** for the CDK app in this directory.
It does **not** create any resources — `cdk deploy --all` already stood up the four stacks
(`-kb`, `-gateway`, `-agent`, `-dashboards`). Here we:

1. Read the stack **outputs** (KB ids, gateway, agent runtime, dashboards).
2. Drive **differentiated agentic traffic per KB** through the deployed Strands agent
   (financial vs weather prompts) — this exercises the whole path: agent → Gateway →
   `AgenticRetrieveStream` → synthesized, cited answer.
3. Read the agent's own telemetry and **publish the three custom-metric layers** the
   dashboards read: **L3** agentic retrieval quality, **L6** token usage, **L7** eval scores.
4. Open the two **live dashboards** — now populated.

The stacks are plain CloudFormation stacks once deployed, so everything below is read through
`describe_stacks` exactly as it would be for a hand-written template. Nothing here is
CDK-specific: the point is that the *authoring* experience differs, not the deployed result.

> **About the data.** The financial corpus (Octank 10-K) is **synthetic** — Octank is a fictional
> company and the document is model-generated, CC0-licensed sample data (not real financial data).
> The weather corpus (tornadoes report) is a **real, publicly available** U.S. Congressional Research
> Service report ([IF12695](https://sgp.fas.org/crs/misc/IF12695.pdf)) — a public-domain U.S.
> Government work.

The 7-layer taxonomy and where each signal originates is documented in
`02-feature-examples/04-rag-evaluation/03-agentic-rag-evaluation-for-managed-kb.ipynb`; this
notebook reuses those exact helpers (`utils/kb_observability.py`) against the deployed stack.

## 1. Setup

The `utils/` here are a **self-contained copy** (no dependency on the repo root), so this
directory can move to its own repo. Install the few runtime deps if needed.

In [ ]:
%pip install -q boto3 pandas pyyaml 'mcp-proxy-for-aws' 2>/dev/null
print('deps ok')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys, json, time, uuid
import boto3, pandas as pd

os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
sys.path.insert(0, '..')   # the local self-contained utils/ package
from utils import kb_observability as obs
from utils import nb_display as ui
from utils.pricing import compute_cost_usd, quota_consumed

region = boto3.session.Session().region_name or os.environ['AWS_DEFAULT_REGION']
cfn = boto3.client('cloudformation', region_name=region)
agentcore = boto3.client('bedrock-agentcore', region_name=region)
ui.rule('Environment')
ui.ok(f'Region: {region}')

## 2. Read the deployed stack outputs

Everything the driver needs comes from CloudFormation outputs — no hard-coded ids. The stack
names themselves come from `config.py`, the same place the CDK app reads them from, so the
two cannot drift apart.

In [ ]:
# Stack names are derived from the same PROJECT_NAME the CDK app uses, so renaming the
# project in config.py cannot leave this notebook pointing at stale stacks.
from config import EnvSettings   # app root is already on sys.path from the setup cell

PROJECT = EnvSettings.PROJECT_NAME
STACKS = {suffix: f'{PROJECT}-{suffix}' for suffix in ('kb', 'gateway', 'agent', 'dashboards')}

def outputs(stack_name):
    s = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]
    return {o['OutputKey']: o['OutputValue'] for o in s.get('Outputs', [])}

kb_out   = outputs(STACKS['kb'])
gw_out   = outputs(STACKS['gateway'])
ag_out   = outputs(STACKS['agent'])
dash_out = outputs(STACKS['dashboards'])

# Two KBs — the semantic-routing pair.
KBS = {
    'financial': kb_out['FinancialKnowledgeBaseId'],
    'weather':   kb_out['WeatherKnowledgeBaseId'],
}
gateway_id  = gw_out['GatewayId']
agent_arn   = ag_out['AgentRuntimeArn']
agent_id    = ag_out['AgentRuntimeId']

ui.rule('Deployed resources')
for theme, kid in KBS.items(): ui.ok(f'KB {theme}: {kid}')
ui.ok(f'Gateway: {gateway_id}')
ui.ok(f'Agent runtime: {agent_id}')
print()
ui.info('Dashboards:')
print(' ', dash_out['ObservabilityDashboardUrl'])
print(' ', dash_out['KbObservabilityDashboardUrl'])

## 3. Index size — deterministic source-bytes fallback

The KB's native `RawDataSize` metric (`AWS/Bedrock/KnowledgeBases`) is emitted **sporadically**
— a fully-managed KB may go a long time without publishing it, or publish for one KB and not
another, even when both ingested successfully. So for a reliable, immediate per-KB size signal we
sum the **source bytes in S3** (each KB's data lives under its own prefix) and publish it as
`BMKB/Cost` `SourceBytesMB`. The dashboard's *Index size* widget reads this — it fills right away
and doesn't depend on the native metric landing.

In [ ]:
src_bucket = kb_out['SourceBucketName']
s3 = boto3.client('s3', region_name=region)
PREFIXES = {'financial': 'financial/', 'weather': 'weather/'}
cw = boto3.client('cloudwatch', region_name=region)

size_rows = []
for theme, kb_id in KBS.items():
    resp = s3.list_objects_v2(Bucket=src_bucket, Prefix=PREFIXES[theme])
    total_bytes = sum(o['Size'] for o in resp.get('Contents', []))
    mb = round(total_bytes / 1048576, 4)
    cw.put_metric_data(Namespace=obs.COST_NAMESPACE, MetricData=[{
        'MetricName': 'SourceBytesMB', 'Value': float(mb), 'Unit': 'Megabytes',
        'Dimensions': [{'Name': 'KnowledgeBaseId', 'Value': kb_id}]}])
    size_rows.append({'kb': theme, 'objects': len(resp.get('Contents', [])),
                      'bytes': total_bytes, 'MB': mb})

ui.rule('Index size (source bytes, per KB)')
print(pd.DataFrame(size_rows))
ui.ok(f'Published SourceBytesMB to {obs.COST_NAMESPACE}')

## 4. Drive differentiated agentic traffic per KB

We send topic-specific prompts so the agent routes each to the matching KB tool
(`financial-agentic___AgenticRetrieveStream` vs `weather-agentic___…`). Each invocation
runs the full agentic loop and is auto-instrumented with OTEL spans by the Runtime.

**`runtimeSessionId` must be ≥ 33 chars** — it is the join key across every layer. We use
**one session per KB** so the per-KB metrics below are cleanly separable.

In [ ]:
PROMPTS = {
    'financial': [
        "What was Octank Financial's total revenue in 2021?",
        "What are Octank's main risk factors?",
        "Describe Octank's growth strategy.",
    ],
    'weather': [
        'How do tornadoes form?',
        'Where do tornadoes most commonly occur?',
    ],
}

# One session id per KB (>= 33 chars). These tie all layers back to each KB's traffic.
SESSIONS = {theme: f'bmkb-obs-{theme}-' + uuid.uuid4().hex + uuid.uuid4().hex[:8]
            for theme in KBS}
for theme, sid in SESSIONS.items():
    ui.ok(f'{theme} session: {sid}  (len={len(sid)})')

def invoke(prompt, session_id):
    r = agentcore.invoke_agent_runtime(
        agentRuntimeArn=agent_arn, qualifier='DEFAULT', runtimeSessionId=session_id,
        payload=json.dumps({'prompt': prompt}).encode(),
        contentType='application/json', accept='application/json')
    body = r['response'].read() if hasattr(r['response'], 'read') else b''.join(r['response'])
    try: return json.loads(body.decode('utf-8'))
    except Exception: return body.decode('utf-8', 'replace')

ui.rule('Driving traffic')
for theme, prompts in PROMPTS.items():
    for q in prompts:
        ans = invoke(q, SESSIONS[theme])
        ui.ok(f'[{theme}] {q[:42]:42}  →  {str(ans)[:60]}...')

ui.info('Waiting 120s for metrics + spans to land (they lag emission by 1–5 min)...')
time.sleep(120)

## 5. Layer 3 — Agentic retrieval quality (per KB)

For each KB we read what the agent **actually retrieved** (from its runtime log,
correlated to the session by `traceId`) and compute the **reference-free** quality signals:
`retrieval_utilization` (precision proxy), `grounded_coverage` (faithfulness proxy),
`duplicate_rate`, plus the retrieval-set counts. No second retrieval call, no ground truth.
We publish them to `BMKB/RetrievalQuality` (dim `KnowledgeBaseId`) — this fills the two
per-KB **L3** widgets on the observability dashboard.

In [ ]:
all_spans = obs.query_spans(region_name=region, hours=1, limit=400)

l3_rows = []
for theme, kb_id in KBS.items():
    sid = SESSIONS[theme]
    trace_ids = {s.get('traceId') for s in all_spans
                 if s.get('attributes', {}).get('session.id') == sid and s.get('traceId')}
    retrievals = obs.fetch_agentic_retrievals(agent_id, trace_ids, region_name=region, hours=1)
    if not retrievals:
        ui.info(f'[{theme}] no agentic-retrieve payloads yet — re-run in ~1 min.')
        continue
    for rp in retrievals:
        stats = obs.extract_agentic_retrieval_quality(rp)
        obs.emit_retrieval_metrics(stats, kb_id=kb_id, region_name=region)   # -> BMKB/RetrievalQuality
        l3_rows.append({'kb': theme, **stats})

if l3_rows:
    df = pd.DataFrame(l3_rows)
    display(df)
    g = df.groupby('kb')[['retrieval_utilization', 'grounded_coverage', 'duplicate_rate']].mean().round(3)
    ui.ok('Per-KB means:'); display(g)
    ui.ok(f'Published L3 to {obs.RETRIEVAL_QUALITY_NAMESPACE}')

## 6. Layers 5 & 6 — Spans and token usage (per KB)

Layer 5 is the agent's span tree (`aws/spans`, auto-emitted). Layer 6 reads the
`gen_ai.usage.*` tokens off the **`chat` CLIENT** spans (the leaf LLM calls — summing all
token-bearing spans would double-count). We publish per-KB **token usage** to `BMKB/Cost`
(`SessionQuotaTokens`) so the **L6** widget fills. (`compute_cost_usd` is available too, but
this blog reports usage, not dollars.)

In [ ]:
def _first(a, *keys):
    for k in keys:
        if a.get(k) not in (None, ''): return a[k]
    return None

l6_rows = []
for theme, kb_id in KBS.items():
    sid = SESSIONS[theme]
    session_tokens = 0
    for s in all_spans:
        a = s.get('attributes', {})
        if a.get('session.id') != sid: continue
        if s.get('kind') == 'CLIENT' and s.get('name', '').startswith('chat'):
            it = _first(a, 'gen_ai.usage.input_tokens', 'gen_ai.usage.prompt_tokens')
            ot = _first(a, 'gen_ai.usage.output_tokens', 'gen_ai.usage.completion_tokens')
            m  = _first(a, 'gen_ai.request.model') or 'anthropic.claude'
            if it is not None and ot is not None:
                session_tokens += quota_consumed(m, int(it), int(ot))
                l6_rows.append({'kb': theme, 'model': m.split('.')[-1],
                                'in_tok': int(it), 'out_tok': int(ot)})
    if session_tokens:
        # Publish token usage per KB. cost_usd is set to 0.0 — this blog reports usage, not price.
        obs.emit_cost_metrics(cost_usd=0.0, quota_tokens=session_tokens,
                              kb_id=kb_id, region_name=region)   # -> BMKB/Cost SessionQuotaTokens
        ui.ok(f'[{theme}] session token usage: {session_tokens:,}  (published to {obs.COST_NAMESPACE})')

if l6_rows:
    display(pd.DataFrame(l6_rows))

## 7. Layer 7 — Quality scores (AgentCore Evaluate, per KB)

For each KB session we download the raw span records and run AgentCore **Evaluate**
(LLM-as-judge) for a few built-in evaluators, then publish to `BMKB/Evaluation` so the
**L7** widget fills. This is the signal a deployed agent unlocks that a local one cannot.

In [ ]:
logs = boto3.client('logs', region_name=region)

def _download_session_spans(session_id):
    def _q(lg):
        start, end = int(time.time() - 3600), int(time.time())
        query = ('fields @timestamp, @message | filter ispresent(scope.name) '
                 f'and ispresent(attributes.session.id) | filter attributes.session.id = "{session_id}" '
                 '| sort @timestamp asc')
        try:
            qid = logs.start_query(logGroupName=lg, startTime=start, endTime=end, queryString=query)['queryId']
        except logs.exceptions.ResourceNotFoundException:
            return []
        while (res := logs.get_query_results(queryId=qid))['status'] not in ('Complete', 'Failed'):
            time.sleep(1)
        rows = res.get('results', []) if res['status'] == 'Complete' else []
        out = []
        for row in rows:
            for f in row:
                if f['field'] == '@message' and f['value'].strip().startswith('{'):
                    try: out.append(json.loads(f['value']))
                    except Exception: pass
        return out
    runtime_lg = f'/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT'
    return _q('aws/spans') + _q(runtime_lg)

EVALUATORS = ['Builtin.Correctness', 'Builtin.Faithfulness', 'Builtin.ToolSelectionAccuracy']
l7_rows = []
for theme, kb_id in KBS.items():
    span_logs = _download_session_spans(SESSIONS[theme])
    if not span_logs:
        ui.info(f'[{theme}] no span records yet — wait 1–2 min and re-run.'); continue
    scores = []
    for ev in EVALUATORS:
        try:
            resp = agentcore.evaluate(evaluatorId=ev, evaluationInput={'sessionSpans': span_logs})
            for r in resp.get('evaluationResults', []):
                scores.append({'evaluator': ev.split('.')[-1], 'score': r.get('value'),
                               'label': r.get('label', ''), 'error': r.get('errorCode', '')})
        except Exception as e:
            scores.append({'evaluator': ev.split('.')[-1], 'score': None, 'error': str(e)[:60]})
    ok = [s for s in scores if s.get('score') is not None]
    if ok:
        obs.emit_eval_scores(scores, kb_id=kb_id, region_name=region)   # -> BMKB/Evaluation
        ui.ok(f'[{theme}] {len(ok)} scores | mean {sum(s["score"] for s in ok)/len(ok):.2f} '
              f'| published to {obs.EVAL_NAMESPACE}')
    for s in scores: s['kb'] = theme
    l7_rows += scores

if l7_rows: display(pd.DataFrame(l7_rows))

## 8. Open the live dashboards

Both dashboards were created by the dashboards stack and now carry data across all seven
layers. Open them, set a **3-hour** range, and refresh (values lag emission by 1–5 min).

The third link is the built-in **GenAI Observability** console view for this agent runtime —
AWS's own rendering of the L5/L6 span data, alongside the custom dashboards.

In [ ]:
ui.rule('Dashboards (now populated)')
print('A · End-to-end agentic observability (7 layers):')
print('   ', dash_out['ObservabilityDashboardUrl'])
print('B · Per-KB (BMKB) observability:')
print('   ', dash_out['KbObservabilityDashboardUrl'])
print('C · GenAI Observability (built-in agent view):')
print('   ', dash_out['GenAiObservabilityUrl'])

## 9. Cleanup

This notebook created **no** resources, so there is nothing to tear down here. To remove
everything, one command from the app directory:

```bash
cdk destroy --all
```

CDK deletes the stacks in reverse dependency order on its own — no teardown script to keep in
sync with a deploy script. The S3 source bucket empties itself (`auto_delete_objects`) and the
ECR repository empties itself (`empty_on_delete`), so the destroy does not stall on non-empty
resources. The custom metrics in `BMKB/*` expire on their own retention.